In [3]:
## non-vectorised
#def _robustify(a0, a, b, T, h_plus, h_minus):
#    """Enforce  a0 + a^T xi <= b  for all xi in the box, by adding the dual certificate
#        mu+ , mu- >= 0 ,  mu+ - mu- == a ,  h+^T mu+ + h-^T mu- <= b - a0.
#    `a0` is a scalar expression, `a` a (T,) expression, `b` a scalar constant.
#    """
#    mu_p = cp.Variable(T, nonneg=True)
#    mu_m = cp.Variable(T, nonneg=True)
#    return [
#        mu_p - mu_m == a,  # H^T mu = a
#        h_plus @ mu_p + h_minus @ mu_m <= b - a0,  # h^T mu <= b - a0


## inside build function
"""
#define soc constraints
    for t in range(T):
        d_ch = cp.reshape(D_ch[t, :], (T,), order = "C") # rows = coefficient vectors on xi
        d_dis = cp.reshape(D_dis[t, :], (T,), order = "C") # rows = coefficient vectors on xi
        g_t = cp.reshape(G[t, :], (T,), order = "C")

        # (1)(2) Charge rate: 0 <= p_ch_t(xi) <= C_ch
        cons += _robustify( p_ch_hat[t],  d_ch, fixed_params.C_ch,  T, h_plus, h_minus)
        cons += _robustify(-p_ch_hat[t], -d_ch, 0.0,                T, h_plus, h_minus)

        # (3)(4) Discharge rate: 0 <= p_dis_t(xi) <= C_dis
        cons += _robustify( p_dis_hat[t], d_dis, fixed_params.C_dis,    T, h_plus, h_minus)
        cons += _robustify(-p_dis_hat[t], -d_dis, 0.0,                  T, h_plus, h_minus)

        # (5)(6) State of Charge: 0 <= s_t(xi) <= B_max
        cons += _robustify( s_hat[t],  g_t, fixed_params.B_max, T, h_plus, h_minus)
        cons += _robustify(-s_hat[t], -g_t, 0.0,                T, h_plus, h_minus)

"""


'\n#define soc constraints\n    for t in range(T):\n        d_ch = cp.reshape(D_ch[t, :], (T,), order = "C") # rows = coefficient vectors on xi\n        d_dis = cp.reshape(D_dis[t, :], (T,), order = "C") # rows = coefficient vectors on xi\n        g_t = cp.reshape(G[t, :], (T,), order = "C")\n\n        # (1)(2) Charge rate: 0 <= p_ch_t(xi) <= C_ch\n        cons += _robustify( p_ch_hat[t],  d_ch, fixed_params.C_ch,  T, h_plus, h_minus)\n        cons += _robustify(-p_ch_hat[t], -d_ch, 0.0,                T, h_plus, h_minus)\n\n        # (3)(4) Discharge rate: 0 <= p_dis_t(xi) <= C_dis\n        cons += _robustify( p_dis_hat[t], d_dis, fixed_params.C_dis,    T, h_plus, h_minus)\n        cons += _robustify(-p_dis_hat[t], -d_dis, 0.0,                  T, h_plus, h_minus)\n\n        # (5)(6) State of Charge: 0 <= s_t(xi) <= B_max\n        cons += _robustify( s_hat[t],  g_t, fixed_params.B_max, T, h_plus, h_minus)\n        cons += _robustify(-s_hat[t], -g_t, 0.0,                T, h_plus, h_mi

In [ ]:
from __future__ import annotations
from itertools import product
import warnings
from dataclasses import dataclass, field

import numpy as np
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer

import torch

@dataclass
class FixedParams:
    T_total: int  # number of t steps in time window T
    num_scenarios: int # number of scenarios being fed into the optimiser
    dt: float  # length of time step t
    eta_ch: float  # charging efficiency
    eta_dis: float  # discharging efficiency
    C_ch: float  # max charge rate
    C_dis: float  # max discharge rate
    B_max: float  # battery energy capacity
    SOC0: float  # initial == terminal state of charge
    k: float = 0.5  # 0 = pure DA arbitrage, 1 = pure imbalance minimisation
    gamma: float = 10**-6  # Tikhonov regularisation coefficient


def _robustify_vec(A0, A, B, T, h_plus, h_minus):
    """
    Vectorized robustification.
    A0: (T,) expression
    A: (T, T) expression
    B: scalar
    """
    # Create (T, T) variables instead of looping to create T variables of size T
    mu_p = cp.Variable((T, T), nonneg=True)
    mu_m = cp.Variable((T, T), nonneg=True)
    
    return [
        mu_p - mu_m == A,  # Evaluates equality for the entire (T,T) matrix
        # (T,T) @ (T,) yields a (T,) vector of dot products for each time step
        mu_p @ h_plus + mu_m @ h_minus <= B - A0, 
    ]

def build_robust(fixed_params, double_imb = False):

    # only fixed params I add in just to make it easier to see what's going on.
    T = fixed_params.T_total
    num_scenarios = fixed_params.num_scenarios
  

    # Variables (decision)
    p_ch_hat  = cp.Variable(T, name="p_ch_hat")
    p_dis_hat = cp.Variable(T, name="p_dis_hat")
    D_ch      = cp.Variable((T, T), name="D_ch")
    D_dis     = cp.Variable((T, T), name="D_dis")

    # Epigraph split of imbalance
    p_plus  = cp.Variable((T, num_scenarios), nonneg=True, name="p_plus")
    p_minus = cp.Variable((T, num_scenarios), nonneg=True, name="p_minus")


    # Parameters (can change for each window)
    xi_samples      = cp.Parameter((num_scenarios, T))      # scenarios
    Sigma_xi_chol   = cp.Parameter((T, T))      # sqrt of second moment of xi_samples
    # pl_hat          = cp.Parameter(T)
    pi_da           = cp.Parameter(T)
    up_reg_cost     = cp.Parameter(T, nonneg=True, name="up_reg_cost")   # lambda^up  (>= 0)
    down_reg_cost   = cp.Parameter(T, nonneg=True, name="down_reg_cost")   # lambda^dn  (>= 0)
    h_plus          = cp.Parameter(T, nonneg=True)
    h_minus         = cp.Parameter(T, nonneg=True)

    # Constraints
    cons = []
    cons += [cp.upper_tri(D_ch) == 0, cp.upper_tri(D_dis) == 0]



    
    ## MAKE SOC EQUATION
    ### NOTES ON ROBUSTIFYING PROBLEM FORMULATION
    # To robustify the constraints of the model, we need to separate the random variable \xi
    # from the rest of the model.
    # To do this, we can represent the total SOC function as an affine function.
    # To do this, we separate the day-ahead decisions (which don't rely on \xi at all) from the
    # recourse actions (which depend on \xi).
    # That means that:
    #   e_soc_t = e_soc_t-1 + dt(eta_ch @ (p_ch_hat_t + D_ch_t @ \xi_t)
    #                                      - (1/eta_dis) *
    #                                      (p_dis_hat_t + D_dis_t @ \xi_t)
    #   e_soc = SOC0 + dt * L @ (eta_ch @ p_ch_hat - (1/eta_dis) *
    #                                           (p_dis_hat)
    #           + dt * L @ (eta_ch @ D_ch - (1/eta_dis) * D_dis) @ xi
    # where L is a T x T lower triangular matrix.
    # (this can be coded as L = np.tril(np.ones((P.T_total, P.T_total))))
    #
    # The parts of the equation independent of xi can be interpreted as the intercept of the
    # SOC equation, whilst the parts that relate to xi can be interpreted as the gradient.
    # So:
    # intercept vector (T,):        s_hat = SOC0 + dt * L @ (eta_ch*p_ch_hat - (1/eta_dis)*p_dis_hat)
    # slope matrix (T,T), row t = g_t:   G = dt * L @ (eta_ch*D_ch - (1/eta_dis)*D_dis)
    # then e_soc(xi) = s_hat + G @ xi    for every xi

    # Note: s_hat can be further broken down into SOC0 + expected_power_flow_per_t * cp.cumsum(expected_power_flow_per_t)
    # 1. Nominal State of Charge (s_hat) - 1D Vector Cumulative Sum
    power_flow_hat = fixed_params.eta_ch * p_ch_hat - (1 / fixed_params.eta_dis) * p_dis_hat # power flow per timestep
    s_hat = fixed_params.SOC0 + fixed_params.dt * cp.cumsum(power_flow_hat)

    # 2. Recourse Gain Matrix (G) - 2D Matrix Cumulative Sum
    D_net = fixed_params.eta_ch * D_ch - (1 / fixed_params.eta_dis) * D_dis # net recourse per timestep
    G = fixed_params.dt * cp.cumsum(D_net, axis=0)  # Cumsum down columns (along time axis t)
        
    # --- VECTORIZED SOC CONSTRAINTS (Replaces the for-loop) ---

    # (1)(2) Charge rate: 0 <= p_ch_t(xi) <= C_ch
    cons += _robustify_vec( p_ch_hat,  D_ch, fixed_params.C_ch, T, h_plus, h_minus)
    cons += _robustify_vec(-p_ch_hat, -D_ch, 0.0,               T, h_plus, h_minus)

    # (3)(4) Discharge rate: 0 <= p_dis_t(xi) <= C_dis
    cons += _robustify_vec( p_dis_hat,  D_dis, fixed_params.C_dis, T, h_plus, h_minus)
    cons += _robustify_vec(-p_dis_hat, -D_dis, 0.0,                T, h_plus, h_minus)

    # (5)(6) State of Charge: 0 <= s_t(xi) <= B_max
    cons += _robustify_vec( s_hat,  G, fixed_params.B_max, T, h_plus, h_minus)
    cons += _robustify_vec(-s_hat, -G, 0.0,                T, h_plus, h_minus)

    # terminal equality. Outside for loop.
    cons += [
        s_hat[T - 1] == fixed_params.SOC0,  # s_hat = SOC0
        G[T - 1, :] == 0,
    ]  # g_T = 0 : recourse is energy-neutral

    ## OBJECTIVE FUNCTION

    # imbalance power recourse expression:
    I_T = np.eye(T)
    recourse_matrix = I_T + D_ch - D_dis # (T,T)

    ## The major ball ache: DPP COMPLIANCE
    # To pass a cvxpy problem through cvxpylayers, the problem must be DPP compliant.
    # """
    # To compile a parameterized problem, CVXPY must be able to express all constraints and objectives in a canonical form (e.g., Ax < b) 
    # where the coefficients (like A and b) are strictly affine functions of the parameters. 
    # Multiplying a cp.Parameter by another cp.Parameter creates a non-linear (quadratic or bilinear) parameter dependence, which CVXPY cannot factorize.
    # """

    # Dealing with this: 
    ## 1. The quadratic deviation term --------------------------------------------------------------------

    # To maintain DPP in this problem, I have to make sure I don't multiply any parameters by any parameters.
    # This means I can't multiply p_imb by itself as I defined originally in my equations, as p_imb contained xi_samples, a PARAMETER.
    # To get around this, I can reformulate the quadratic part of the equation using the Georghiou et. al. paper original LDR paper.
    # E[(r_t @ xi)^2] = r_t @ Sigma_xi @ r_t^T
    # where r_t is the t'th row of the recouse matrix, Sigma_xi is the second moment of xi_samples.
    # This would be:
    # sum_trace = fixed_params.dt**2 * sum(cp.quad_form(recourse_matrix[t, :], Sigma_xi) for t in range(T))
    # HOWEVER:
    # apparently cvxpy doesn't like that formulation and it wouldn't be DPP, so I have to do it using the cholesky factor and using sum_squares.
    sum_trace = fixed_params.dt**2 * cp.sum_squares(recourse_matrix @ Sigma_xi_chol)

    
    ## 2. The economic term -------------------------------------------------------------------------------

    # Day ahead cost
    # C_da = pi^da.T p^bid . dt
    # p^imb = pl_hat + p^ch - p^dis
    # C_da = pi^da.T @ (pl_hat + p^ch − p^dis) . dt
    # This equation expands into two part:
    # C_da = ...
    # pi^da @ (p^ch − p^dis) . dt <-- this part is DPP compliant
    # + pi^da @ pl_hat . dt         <-- this part isn't. both pi^da and pl_hat are params.
    # However, this second term  is constant. 
    # It doesn't actually affect the best decision, it adds nothing of use to the problem.
    # This means it can be removed from the optimisation problem and added back in after.
    # Therefore:
    C_da = pi_da @ (p_ch_hat - p_dis_hat) * fixed_params.dt      # variable part only, DPP compliant.

    # Imbalance cost

    # one way to implement: 
    # C_imb = sum(up_reg_cost @ cp.pos(p_imb) + down_reg_cost @ cp.neg(p_imb)) * dt / N
    # where p_imb = recourse_matrix @ xi_samples.T
    # However, DPP does not accept non-linear atoms (like .pos and .neg) that are not purely a variable or purely a parameter.
    # Also, multiplying an imbalance cost by p_imb means that it must be param(price) X var (rec matrix) X param (xi_samples).
    # Because of both of these reasons, it is not DPP compliant.

    # However, p_imb can be reformulated using an epigraph:
    # p_plus  = cp.Variable((T, N), nonneg=True, name="p_plus")
    # p_minus = cp.Variable((T, N), nonneg=True, name="p_minus")
    # where p_plus and p_minus equals p_imb at all timesteps.
    # This makes it DPP compliant!

    # For each pos, neg of imbalance cost you get:
    # cost @ positive_imbalance
    # this is a param X var, which is DPP compliant.
    # problem solved!!

    p_imb = recourse_matrix @ xi_samples.T  # Result: (T, N)
    cons += [p_plus - p_minus == p_imb]     # affine in xi  => DPP-ok
    
    C_imb = cp.sum(up_reg_cost @ p_plus + down_reg_cost @ p_minus) * fixed_params.dt / num_scenarios

    ## 3. Penalty term --------------------------------------------------------------------------------------
    # This is to try and produce a gradient at all possible solutions, by adding a tiny penalty term to all decision vars.
    penalty = fixed_params.gamma * (
        cp.sum_squares(p_ch_hat) + cp.sum_squares(p_dis_hat)
        + cp.sum_squares(D_ch) + cp.sum_squares(D_dis)
    )

    ## 4. Put it all together -------------------------------------------------------------------------------
    econ = C_da + C_imb

    obj = cp.Minimize((1 - fixed_params.k) * econ + fixed_params.k * sum_trace + penalty)


    prob = cp.Problem(obj, cons)

    assert prob.is_dcp(dpp=True), "not DPP — cvxpylayers will reject"

    layer = CvxpyLayer(prob,
                       parameters=[xi_samples, Sigma_xi_chol, pi_da, up_reg_cost, down_reg_cost, h_plus, h_minus],
                       variables=[p_ch_hat, p_dis_hat, D_ch, D_dis])   # what you want back
    return layer

def build_oracle(fp: FixedParams):
    """Perfect-foresight deterministic dispatch, with imbalance arbitrage allowed.

    The bid is a free decision (not forced to equal realised net draw), so the oracle may
    deliberately deviate to exploit the imbalance market. It faces the SAME imbalance cost
    structure as the forecast-driven policy (lam_up/lam_dn, no-arbitrage penalty), so with
    a genuine dual price it will optimally choose zero imbalance; with a signed/favourable
    price it can arbitrage. This makes the oracle a fair upper bound: it has every option
    the policy has, plus perfect foresight.
    """
    T = fp.T_total
    p_ch  = cp.Variable(T, nonneg=True)
    p_dis = cp.Variable(T, nonneg=True)
    bid   = cp.Variable(T)                       # FREE bid (the arbitrage lever)
    p_plus  = cp.Variable(T, nonneg=True)        # imbalance epigraph split
    p_minus = cp.Variable(T, nonneg=True)

    p_d             = cp.Parameter(T)
    pi_da           = cp.Parameter(T)
    up_reg_cost     = cp.Parameter(T, nonneg=True)
    down_reg_cost   = cp.Parameter(T, nonneg=True)

    L = np.tril(np.ones((T, T)))
    soc = fp.SOC0 + fp.dt * (L @ (fp.eta_ch * p_ch - (1.0 / fp.eta_dis) * p_dis))
    cons = [p_ch <= fp.C_ch, p_dis <= fp.C_dis, soc >= 0, soc <= fp.B_max]
    if fp.terminal_soc_equality:
        cons += [soc[T - 1] == fp.SOC0]
    else:
        cons += [soc[T - 1] >= fp.SOC0]

    # imbalance = realised net draw - bid, split for the asymmetric cost
    net_draw = p_d + p_ch - p_dis
    cons += [p_plus - p_minus == net_draw - bid]

    C_da  = pi_da @ bid * fp.dt
    C_imb = (up_reg_cost @ p_plus + down_reg_cost @ p_minus) * fp.dt
    prob = cp.Problem(cp.Minimize(C_da + C_imb), cons)

    def solve(p_d_val, pi_da_val, lam_up_val, lam_dn_val, solver=cp.CLARABEL):
        p_d.value           = np.asarray(p_d_val, float)
        pi_da.value         = np.asarray(pi_da_val, float)
        up_reg_cost.value   = np.asarray(lam_up_val, float)
        down_reg_cost.value = np.asarray(lam_dn_val, float)
        prob.solve(solver=solver)
        return float(prob.value)
    return solve

In [16]:
T = 24
N = 64

fp = FixedParams(
    T_total=T, num_scenarios=N, dt=1.0,
    eta_ch=0.95, eta_dis=0.95, C_ch=2.0, C_dis=2.0,
    B_max=6.0, SOC0=3.0, k = 0, gamma=1e-6
)
model = build_robust(fp)

In [6]:


p_mean = sampler.prosumption(quantiles).mean(dim=0)   # (K,) mean forecast per lead
h_plus  = q_upper  - p_mean      # how far up ξ can go from the mean
h_minus = p_mean   - q_lower     # how far down ξ can go from the mean

NameError: name 'sampler' is not defined

In [ ]:
# --------------------------------------------------------------------------
def _to_t(a, like=None, dtype=torch.double):
    """Coerce numpy/torch/scalar to a torch tensor (double by default)."""
    if isinstance(a, torch.Tensor):
        t = a.to(dtype)
    else:
        t = torch.as_tensor(np.asarray(a), dtype=dtype)
    if like is not None:
        t = t.to(like.device)
    return t


@dataclass
class DispatchInputs:
    xi_samples:     torch.Tensor   # (N, T)  mean-centred error scenarios       (requires grad)
    L_xi:           torch.Tensor   # (T, T)  cholesky decomp. of xi_samples cov.(requires grad)
    pl_hat:         torch.Tensor   # (T,)    mean forecast                      (requires grad)
    pi_da:          torch.Tensor   # (T,)
    up_reg_cost:    torch.Tensor   # (T,)
    down_reg_cost:  torch.Tensor   # (T,)
    h_plus:         torch.Tensor   # (T,)    box half-width up   (detached by default)
    h_minus:        torch.Tensor   # (T,)    box half-width down (detached by default)

    def layer_args(self):
        """Positional tensors for CvxpyLayer, in the order build_layer expects:
        [xi_samples, pi_da, lam_up, lam_dn, h_plus, h_minus]."""
        return (self.xi_samples, self.L_xi, self.pi_da, self.lam_up, self.lam_dn,
                self.h_plus, self.h_minus)

def make_dispatch_inputs(
    quantiles,                 # (K, Q) monotone quantiles for ONE day, physical MW, torch
    sampler,                   # FrozenCopulaSampler  (mean_and_errors)
    quantile_levels,           # (Q,)  the forecaster's levels (e.g. 0.05..0.95)
    pi_da,                     # (T,)  day-ahead prices        (from window loader)
    up_reg_cost,               # (T,)  lambda^up  >= 0          (pre-saved, from loader)
    down_reg_cost,             # (T,)  lambda^dn  >= 0          (pre-saved, from loader)
    box_levels=(0.05, 0.95),   # (lower, upper) forecast quantiles defining the box edges
    box_detach=True,           # detach h_plus/h_minus so DFL cannot game the robustness margin
    min_box=1e-4,              # floor on box half-widths (guards extreme-skew negatives)
):
    """Build mean-anchored, mutually-consistent dispatch inputs from a day's quantiles.

    The box edges come from *specific quantiles* (box_levels) but are expressed as offsets
    from the MEAN, so the box lives in the same xi = (realised - mean) coordinate as the
    error scenarios and the imbalance:
        h_plus  = q_upper - mean      (how far up   xi may reach)
        h_minus = mean    - q_lower   (how far down xi may reach)
    """
    if quantiles.dim() != 2:
        raise ValueError("quantiles must be (K, Q) for a single day")

    # mean forecast and mean-centred error scenarios, from ONE prosumption pass
    mean, xi = sampler.mean_and_errors(quantiles)      # mean (K,), xi (S, K) = (N, T)
    cov_xi = cov_xi = torch.cov(xi.T)

    try:
        # Use PyTorch's native Cholesky decomposition
        L_xi = torch.linalg.cholesky(cov_xi)
    except torch.linalg.LinAlgError:
        print("Cholesky decomp didn't work. add 1e-6 to trace to force positive definite")
        jitter = 1e-6
        # Create the identity matrix using the same device and dtype as cov_xi
        regularised_cov = cov_xi + (jitter * torch.eye(cov_xi.shape[0], dtype=cov_xi.dtype, device=cov_xi.device))
        L_xi = torch.linalg.cholesky(regularised_cov)
    
    pl_hat = mean                                       # (T,) anchor; grad kept

    # locate the box-edge quantiles in the level grid
    levels = np.asarray(quantile_levels, float)
    i_lo = int(np.argmin(np.abs(levels - box_levels[0]))) 
    i_hi = int(np.argmin(np.abs(levels - box_levels[1])))
    q_lower = quantiles[:, i_lo]                        # (T,)
    q_upper = quantiles[:, i_hi]                        # (T,)

    # box half-widths expressed against the MEAN
    h_plus  = q_upper - mean                            # (T,)
    h_minus = mean - q_lower                            # (T,)

    # guard: extreme lower/upper-tail skew can (rarely) push the mean outside [q_lo, q_hi],
    # which would make a half-width negative. Clamp and warn -- a non-positive half-width
    # means the box is degenerate on that side.
    with torch.no_grad():
        if (h_plus < 0).any() or (h_minus < 0).any():
            warnings.warn(
                "mean fell outside [q_lower, q_upper] on some lead(s) (extreme skew); "
                "clamping box half-width to min_box. Consider wider box_levels.")
    h_plus  = torch.clamp(h_plus,  min=min_box)
    h_minus = torch.clamp(h_minus, min=min_box)

    if box_detach:
        # robustness margin is a per-day spec derived from the forecast, but NOT a quantity
        # the decision loss should optimise away -> detach so no gradient flows through h.
        h_plus  = h_plus.detach()
        h_minus = h_minus.detach()

    return DispatchInputs(
        xi_samples=xi,                                  # (N, T), grad
        L_xi=L_xi,                                      # (T, T), grad
        pl_hat=pl_hat,                                  # (T,),   grad
        pi_da=_to_t(pi_da,  like=xi),
        lam_up=_to_t(up_reg_cost, like=xi),
        lam_dn=_to_t(down_reg_cost, like=xi),
        h_plus=_to_t(h_plus,  like=xi),
        h_minus=_to_t(h_minus, like=xi),
    )

def realised_cost(
    fp: FixedParams,
    p_ch_hat, p_dis_hat, D_ch, D_dis,   # decision tensors from the layer (grad)
    realised,                           # (T,) realised prosumption for the day
    pl_hat,                             # (T,) mean forecast (same anchor as inputs)
    pi_da, lam_up, lam_dn,              # (T,) prices
    clip_recourse=False,                # clip battery actions to physical limits (kinks)
):
    """True economic cost of the layer's decision at the realised prosumption.

    Differentiable in the decision variables and pl_hat (hence in the forecaster). This is
    the DFL loss; regret() subtracts the (detached) oracle.

    Note: the pi_da @ pl_hat term that was DROPPED from the layer objective is added back
    here (it is part of the real DA cost, just not decision-relevant).
    """
    dev = p_ch_hat.device
    T = fp.T_total
    I_T = torch.eye(T, dtype=p_ch_hat.dtype, device=dev)

    realised = _to_t(realised, like=p_ch_hat)
    pl_hat   = _to_t(pl_hat,   like=p_ch_hat)
    pi_da    = _to_t(pi_da,    like=p_ch_hat)
    lam_up   = _to_t(lam_up,   like=p_ch_hat)
    lam_dn   = _to_t(lam_dn,   like=p_ch_hat)

    xi_real = realised - pl_hat                         # (T,) mean-centred realised error

    # committed day-ahead bid (here-and-now decisions; recourse is not in the bid)    
    bid = pl_hat + p_ch_hat - p_dis_hat                 # (T,)

    # realised imbalance:  p_imb = (I + D_ch - D_dis) @ xi_real
    # (if clipping, recompute from the saturated actions so imbalance reflects saturation)
    # clip based on stratigakos prescriptive trees paper.
    if clip_recourse:
        p_soc = fp.SOC0
        p_ch_clipped, p_dis_clipped = [], []
        for t in range(T):
            pc = p_ch_hat[t]  + D_ch[t]  @ xi_real     # raw recourse action at t
            pd = p_dis_hat[t] + D_dis[t] @ xi_real
            # state-dependent saturation
            pc = torch.clamp(pc, 0.0, torch.clamp(min(fp.C_ch), min=0.0))

            max_ch  = torch.minimum(torch.tensor(fp.C_ch),  (fp.B_max - p_soc) / fp.eta_ch)
            max_dis = torch.minimum(torch.tensor(fp.C_dis), p_soc * fp.eta_dis)

            pc = torch.clamp(pc, min=0.0)
            pc = torch.minimum(pc, torch.clamp(max_ch, min=0.0))

            pd = torch.clamp(pd, min=0.0)
            pd = torch.minimum(pd, torch.clamp(max_dis, min=0.0))
            # advance SOC with the SATURATED action
            p_soc = p_soc + fp.dt * (fp.eta_ch * pc - (1.0/fp.eta_dis) * pd)
            p_ch_clipped.append(pc)
            p_dis_clipped.append(pd)

        p_ch_r  = torch.stack(p_ch_clipped)
        p_dis_r = torch.stack(p_dis_clipped)
    
        # net realised draw minus committed bid
        net_draw = realised + p_ch_r - p_dis_r
        p_imb = net_draw - bid
    else:
        p_imb = (I_T + D_ch - D_dis) @ xi_real          # (T,)

    C_da  = (pi_da * bid).sum() * fp.dt                 # includes pi_da @ pl_hat
    C_imb = (lam_up * torch.clamp(p_imb, min=0.0)
             + lam_dn * torch.clamp(-p_imb, min=0.0)).sum() * fp.dt
    return C_da + C_imb                                 # scalar, grad -> forecaster



def regret(
    fp: FixedParams,
    p_ch_hat, p_dis_hat, D_ch, D_dis,
    realised, pl_hat, pi_da, up_reg_cost, down_reg_cost,
    oracle_solve,                       # closure from build_oracle
    clip_recourse=False,
):
    """Regret = realised_cost(forecast decision) - oracle_cost.

    The oracle term is a detached constant (perfect foresight, no forecast dependence), so
    the gradient of regret w.r.t. the forecaster equals the gradient of realised_cost. Use
    realised_cost as the DFL loss; regret is the reported, interpretable metric (>= 0).
    """
    cost_fcst = realised_cost(
        fp, p_ch_hat, p_dis_hat, D_ch, D_dis,
        realised, pl_hat, pi_da, up_reg_cost, down_reg_cost, clip_recourse=clip_recourse,
    )
    with torch.no_grad():
        oracle_cost = oracle_solve(_to_t(realised).cpu().numpy(),
                                   _to_t(pi_da).cpu().numpy())
    return cost_fcst - oracle_cost      # scalar; grad only through cost_fcst

# Messy as fuck but I just want to prove it works.

In [21]:
import os
import sys
from pathlib import Path
from pyprojroot import here

import copy
import time
import random
import subprocess
import pickle

import numpy as np
import pandas as pd
from scipy.stats import norm
import torch
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
ROOT_DIR = here()
FORECASTING_DIR = ROOT_DIR / "4_forecasting"
DATA_DIR = ROOT_DIR / "1_data" / "processed"
COPULA_SAVE_DIR = ROOT_DIR / "5_scenario_gen"

sys.path.insert(0, str(FORECASTING_DIR))  # make forecasting module importable
sys.path.insert(0, str(COPULA_SAVE_DIR))  # make copula module importable

In [ ]:
# ---- reproducibility (record SEED in the checkpoint) ----
SEED = 20240801
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| seed:", SEED)

In [ ]:
from forecasting import (
    HIST_COLS, FEAT_COLS, EXO_COLS,
    TRAIN_START, VAL_START, TEST_START, TEST_END,
    build_features, reindex_and_impute, make_windows,
    fit_scalers, normalise_hist, normalise_y, denormalise_y,
    Baseline_Forecaster, pinball_loss,
)

In [ ]:
base_data = pd.read_csv(DATA_DIR / "df_full.csv", parse_dates=["datetime"])
base_data.set_index("datetime", inplace=True)

base_data  = reindex_and_impute(base_data, HIST_COLS, freq="1h", warn_gap=6)
frame_full = build_features(base_data, feature_cols=FEAT_COLS)
print("frame_full:", frame_full.shape, "|", frame_full.index.min(), "->", frame_full.index.max())

In [ ]:
def load_forecaster(device):
    """Rebuild the baseline (64) model and load the warm-start weights.
    Returns a torch nn.Module in the given device, with grad enabled on its params.
    """
    from forecasting import Baseline_Forecaster
    ckpt = torch.load(FORECASTING_DIR / "baseline_forecaster_best.pt", weights_only=False, map_location="cpu")
    model = Baseline_Forecaster(**ckpt["model_config"])
    model.load_state_dict(ckpt["state_dict"])
    model.to(device)
    sc = ckpt["scaler_stats"]
    levels = np.asarray(ckpt["quantile_levels"], float)
    return model, sc, levels

def load_sampler(device, rebuild_Z_corr = False, levels = None, n = 64, K = 24, seed = SEED, eps = EPS:):
    """Load the frozen copula bundle and build a FrozenCopulaSampler (buffers only).
    Returns the sampler (on device). Its S must equal FixedParams.num_scenarios.
    """
    from copula import FrozenCopulaSampler, build_Z_corr   # wherever the class lives

    if rebuild_Z_corr:
        assert levels != None, "if rebuilding Z_corr, must provide levels variable!"
        raise
        year_corr = np.load(COPULA_SAVE_DIR / "year_corr_matrix.npy")
        Z_corr = build_Z_corr(year_corr, K = 24,seed = SEED,n = 256, eps = EPS)
    return Frozen_CopulaSampler(Z_corr, levels).to(device)
    
    bundle = pickle.load(open(COPULA_SAVE_DIR / "frozen_copula.pkl", "rb"))
    Z_corr = bundle["Z_corr"]; levels = bundle["quantile_levels"]
    return FrozenCopulaSampler(Z_corr, levels).to(device)
